# Day 4 — Greeks: predicted versus observed

This notebook studies **local sensitivities of the theoretical dividend-adjusted European Black–Scholes price per share**. It uses hypothetical inputs, not market quotes or implied volatility. All calculations are in Python and are independent of production modules.

A Greek is a partial derivative: change one input while holding the others fixed. For small changes, the derivative predicts the option-value change; the error grows as the change becomes larger or the derivative itself changes. Greeks are model sensitivities, not guarantees of actual market-price changes.

Inputs: spot $S$ and strike $K$ in currency/share, time $T$ in years, annual volatility $\sigma$ as a decimal, continuously compounded annual risk-free rate $r$ and dividend yield $q$ as decimals. Let $N$ and $\phi$ be the standard normal CDF and PDF, and

$$d_1=\frac{\ln(S/K)+(r-q+\sigma^2/2)T}{\sigma\sqrt T},\qquad d_2=d_1-\sigma\sqrt T.$$

We use the [Options Industry Council's description of Greeks](https://www.optionseducation.org/advancedconcepts/volatility-the-greeks) for their interpretation and derive the formulas below by differentiating the [Day 3 pricing formula](03_black_scholes.ipynb).

## What each Greek means and its display unit

- **Delta** $\partial V/\partial S$: model-price change for a +1 currency-unit spot move, locally. Call: $e^{-qT}N(d_1)$; put: $-e^{-qT}N(-d_1)$. Calls usually have positive Delta and puts negative Delta. Far in the money, Delta approaches its discounted spot limit ($e^{-qT}$ for calls, $-e^{-qT}$ for puts); far out of the money, it approaches zero. It becomes steeper near the strike as expiry nears.
- **Gamma** $\partial^2 V/\partial S^2=\partial\Delta/\partial S$: Delta change per +1 currency-unit spot move. Both types have $e^{-qT}\phi(d_1)/(S\sigma\sqrt T)>0$. Often largest near the strike, especially close to expiry. It explains why a fixed Delta estimate deteriorates for larger spot moves.
- **Vega** $\partial V/\partial\sigma=S e^{-qT}\phi(d_1)\sqrt T$: model-price change per 1.00 decimal annual-volatility input. We display **$0.01\times$ Vega**, so a displayed 0.38 predicts about +0.38 currency/share for a rise from 20% to 21%, locally. Both call and put Vega are positive for $T>0$ and $\sigma>0$; it is usually largest near the strike and for longer maturities.
- **Theta** $\partial V/\partial t=-\partial V/\partial T$: model-price change as one calendar day passes with spot and other inputs fixed. We display annual Theta divided by 365. The common term is $-S e^{-qT}\phi(d_1)\sigma/(2\sqrt T)$. Add $-rKe^{-rT}N(d_2)+qSe^{-qT}N(d_1)$ for a call, or $+rKe^{-rT}N(-d_2)-qSe^{-qT}N(-d_1)$ for a put. Theta is often negative, especially near-the-money close to expiry, but rates and dividends can make it positive.
- **Rho**, $\partial V/\partial r$, concerns a rate change; it is introduced here for context and is not a Day 4 dashboard output. Its call value is $KTe^{-rT}N(d_2)$ and put value is $-KTe^{-rT}N(-d_2)$ per 1.00 rate change, or 0.01 times that per percentage point. It is usually smaller for short expiries and matters more as time increases.

At expiry and effectively zero volatility, the smooth formulas can be undefined or unstable near a payoff kink. The dashboard reports Greeks as unavailable there. These are **per-share**, not contract or portfolio, quantities. No stock move, volatility move, or day of decay changes only one input in a real market.

In [ ]:
import numpy as np
from scipy.special import ndtr

# Notebook-only formulas; production code does not import this notebook.
def model_price(S, K, T, sigma, r, q, kind):
    d1 = (np.log(S / K) + (r - q + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if kind == "call":
        return S * np.exp(-q * T) * ndtr(d1) - K * np.exp(-r * T) * ndtr(d2)
    return K * np.exp(-r * T) * ndtr(-d2) - S * np.exp(-q * T) * ndtr(-d1)

def local_greeks(S, K, T, sigma, r, q, kind):
    d1 = (np.log(S / K) + (r - q + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    pdf = np.exp(-0.5 * d1**2) / np.sqrt(2 * np.pi)
    ds = S * np.exp(-q * T)
    dk = K * np.exp(-r * T)
    gamma = np.exp(-q * T) * pdf / (S * sigma * np.sqrt(T))
    vega_point = ds * pdf * np.sqrt(T) * 0.01
    common_theta = -ds * pdf * sigma / (2 * np.sqrt(T))
    if kind == "call":
        delta = np.exp(-q * T) * ndtr(d1)
        theta_day = (common_theta - r * dk * ndtr(d2) + q * ds * ndtr(d1)) / 365
    else:
        delta = -np.exp(-q * T) * ndtr(-d1)
        theta_day = (common_theta + r * dk * ndtr(-d2) - q * ds * ndtr(-d1)) / 365
    return dict(delta=delta, gamma=gamma, vega_point=vega_point, theta_day=theta_day)

base = dict(S=100.0, K=100.0, T=1.0, sigma=0.20, r=0.05, q=0.02, kind="call")
g = local_greeks(**base)
print("Call model value:", round(model_price(**base), 6))
print("Call Greeks:", {key: round(value, 6) for key, value in g.items()})
assert np.isclose(model_price(**base), 9.227005508154036)


## One-step predictions versus repricing

Starting with a 100 spot, 100 strike, one year, 20% volatility, 5% rate, and 2% yield, calculate the call and its Greeks above. Predict one change at a time. Delta predicts $\Delta V\approx\Delta\,\Delta S$; Gamma predicts $\Delta\Delta\approx\Gamma\,\Delta S$ and improves the price prediction to $\Delta V\approx\Delta\,\Delta S+\tfrac12\Gamma(\Delta S)^2$. Vega predicts $\Delta V\approx\text{Vega}_{1\%}\,(\Delta\sigma/0.01)$. Theta predicts $\Delta V\approx\text{Theta}_{day}\times\text{days elapsed}$, with $T$ reduced by days/365. “Observed” below means **repriced by the model**, not observed in a market.

In [ ]:
price0 = model_price(**base)

def changed(**overrides):
    return model_price(**(base | overrides)) - price0

for move in [0.10, 1.0, 10.0]:
    predicted = g["delta"] * move
    improved = predicted + 0.5 * g["gamma"] * move**2
    observed = changed(S=base["S"] + move)
    print(f"spot +{move:4.1f}: Delta prediction {predicted:+.6f}, Delta+Gamma {improved:+.6f}, repriced {observed:+.6f}")
    if move == 0.10:
        assert abs(improved - observed) < abs(predicted - observed)

for vol_points in [1, 5]:
    predicted = g["vega_point"] * vol_points
    observed = changed(sigma=base["sigma"] + vol_points * 0.01)
    print(f"vol +{vol_points} point(s): Vega prediction {predicted:+.6f}, repriced {observed:+.6f}")
    if vol_points == 1:
        assert abs(predicted - observed) < 0.01

for days in [1, 30]:
    predicted = g["theta_day"] * days
    observed = changed(T=base["T"] - days / 365)
    print(f"time +{days} day(s): Theta prediction {predicted:+.6f}, repriced {observed:+.6f}")
    if days == 1:
        assert abs(predicted - observed) < 0.0001

spot_step = 0.01
new_delta = local_greeks(**(base | {"S": base["S"] + spot_step}))["delta"]
print(f"Delta after spot +{spot_step}: Gamma prediction {g['delta'] + g['gamma'] * spot_step:.8f}, recomputed {new_delta:.8f}")
assert abs(g["delta"] + g["gamma"] * spot_step - new_delta) < 1e-5


## Moneyness and time experiment

Vary spot and time independently. Notice how Delta approaches its limiting values, how Gamma concentrates around the strike close to expiry, and how Vega shrinks as expiry approaches. Theta can accelerate near the strike; its sign is conditional on rates and dividends. The table is a set of model outputs, not a forecast.

In [ ]:
for T in [1.0, 0.25, 7 / 365]:
    print(f"\nT={T:.4f} years | spot | Delta | Gamma | Vega / vol point | Theta / day")
    for S in [80.0, 100.0, 120.0]:
        values = local_greeks(**(base | {"S": S, "T": T}))
        print(f"{S:5.0f} | {values['delta']:+.4f} | {values['gamma']:.4f} | {values['vega_point']:.4f} | {values['theta_day']:+.4f}")

# Numerical validation of each formula against a small central perturbation.
h, dv, dt = 0.01, 1e-4, 1e-4
finite_delta = (changed(S=100+h) - changed(S=100-h)) / (2*h)
finite_gamma = (changed(S=100+h) - 2*changed() + changed(S=100-h)) / h**2
finite_vega_point = (changed(sigma=0.2+dv) - changed(sigma=0.2-dv)) / (2*dv) * 0.01
finite_theta_day = (changed(T=1-dt) - changed(T=1+dt)) / (2*dt) / 365
for name, expected in [("delta", finite_delta), ("gamma", finite_gamma), ("vega_point", finite_vega_point), ("theta_day", finite_theta_day)]:
    assert np.isclose(g[name], expected, atol=2e-7), (name, g[name], expected)
print("All four analytic Greeks match small finite differences of the notebook price.")
